In [1]:
import torch
import random
import numpy as np
import os

def setupSeed(seed=32):
  os.environ['PYTHONHASHSEED'] = str(seed)
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True

setupSeed()

In [ ]:
!pip install transformers faiss-cpu

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from transformers import AutoModel, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader,Dataset
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from tqdm import tqdm
import faiss
import numpy as np

In [4]:
#----Device----!
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Load data and preprocessing.......


In [5]:
ds = load_dataset("microsoft/ms_marco", "v1.1")

In [6]:
print(ds)

DatasetDict({
    validation: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 10047
    })
    train: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 82326
    })
    test: Dataset({
        features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
        num_rows: 9650
    })
})


In [7]:
train_ds = ds['train']
val_ds = ds['validation']

In [8]:
#--------Train Dataset-------!
train_processed_queries = []
train_processed__positive_passage = []
for i in train_ds:
  query = i['query']
  passages = i['passages']
  for text,selected in zip(passages['passage_text'],passages['is_selected']):
    if selected == 1:
      train_processed_queries.append(query)
      train_processed__positive_passage.append(text)
      break

#--------Validation Dataset-------!
val_processed_queries = []
val_processed__positive_passage = []
for i in val_ds:
  query = i['query']
  passages = i['passages']
  for text,selected in zip(passages['passage_text'],passages['is_selected']):
    if selected == 1:
      val_processed_queries.append(query)
      val_processed__positive_passage.append(text)
      break


Model and Data loader

In [ ]:
autoTokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
autoModel = AutoModel.from_pretrained('bert-base-uncased')

In [10]:
#-----Encoding------!
train_query_encoding = autoTokenizer(text=train_processed_queries,padding=False,max_length=512,truncation=True,add_special_tokens=True,return_attention_mask=True)
train_passage_encoding = autoTokenizer(text=train_processed__positive_passage,padding=False,max_length=512,truncation=True,add_special_tokens=True,return_attention_mask=True)

val_query_encoding = autoTokenizer(text=val_processed_queries,padding=False,max_length=512,truncation=True,add_special_tokens=True,return_attention_mask=True)
val_passage_encoding = autoTokenizer(text=val_processed__positive_passage,padding=False,max_length=512,truncation=True,add_special_tokens=True,return_attention_mask=True)

In [11]:
#-------tokenizing-----!
class TokenAttentionLayer(Dataset):
  def __init__(self,query,passage):
    self.query = query
    self.passage = passage

  def __len__(self):
    return len(self.query['input_ids'])

  def __getitem__(self,idx):
    query_input_ids = self.query['input_ids'][idx]
    query_attention_mask = self.query['attention_mask'][idx]

    passage_input_ids = self.passage['input_ids'][idx]
    passage_attention_mask = self.passage['attention_mask'][idx]

    return {
        "query_input_ids":query_input_ids,
        "query_attention_mask":query_attention_mask,
        "passage_input_ids":passage_input_ids,
        "passage_attention_mask":passage_attention_mask
    }

In [12]:
def retrieval_collate_fn(batch):
    query_input_ids = pad_sequence([torch.as_tensor(x["query_input_ids"]) for x in batch],batch_first=True,padding_value=autoTokenizer.pad_token_id)
    query_attention_mask = pad_sequence([torch.as_tensor(x["query_attention_mask"]) for x in batch],batch_first=True,padding_value=0)

    passage_input_ids = pad_sequence([torch.tensor(x["passage_input_ids"]) for x in batch],batch_first=True,padding_value=autoTokenizer.pad_token_id)

    passage_attention_mask = pad_sequence([torch.tensor(x["passage_attention_mask"]) for x in batch],batch_first=True,padding_value=0)

    return {
        "query_input_ids": query_input_ids,
        "query_attention_mask": query_attention_mask,

        "passage_input_ids": passage_input_ids,
        "passage_attention_mask": passage_attention_mask,
    }

In [13]:
train_dataset = TokenAttentionLayer(train_query_encoding,train_passage_encoding)
val_dataset = TokenAttentionLayer(val_query_encoding,val_passage_encoding)

In [14]:
train_dataLoader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=True,pin_memory=True,collate_fn=retrieval_collate_fn)
val_dataLoader = DataLoader(dataset=val_dataset,batch_size=32,shuffle=False,pin_memory=True,collate_fn=retrieval_collate_fn)

In [15]:
#-----Model class--------!
class BertPretrained(nn.Module):
  def __init__(self,autoModel):
    super().__init__()
    self.autoModel = autoModel
    self.dropout = nn.Dropout(0.2)

  def forward(self,input_ids,attention_mask):
    x = self.autoModel(input_ids=input_ids,attention_mask=attention_mask)
    x = x.last_hidden_state

    mask = attention_mask.unsqueeze(-1).float()
    output = (x*mask).sum(dim=1) / mask.sum(dim=1)

    return self.dropout(output)

In [16]:
bert_model = BertPretrained(autoModel= autoModel)
# for param in bert_model.autoModel.encoder.parameters():
#     param.requires_grad = True

In [17]:
learning_rate = 2e-5
weight_decay = 0.01
temperature = 0.05
epochs = 10

In [18]:
bert_model = bert_model.to(device)
optimizer = optim.AdamW(params=filter(lambda p:p.requires_grad, bert_model.parameters()),lr=learning_rate,weight_decay=weight_decay)
loss_fn = nn.CrossEntropyLoss()
scalar = torch.amp.GradScaler('cuda')

If your notebook is still running and bert_model is already trained, skip this step.

Otherwise:

In [19]:
# checkpoint = torch.load(
#     "best_bert_semantic_retrieval.pth",
#     map_location=device
# )

# bert_model.load_state_dict(checkpoint["model_state_dict"])
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# bert_model.eval()

In [20]:
def generate_passage_embeddings(model, dataloader, device):

    model.eval()

    all_passage_embeddings = []

    with torch.no_grad():
        with torch.amp.autocast("cuda"):

            for batch in tqdm(dataloader, desc="Generating Passage Embeddings"):

                passage_input_ids = batch["passage_input_ids"].to(device)
                passage_attention_mask = batch["passage_attention_mask"].to(device)

                passage_embeddings = model(
                    input_ids=passage_input_ids,
                    attention_mask=passage_attention_mask
                )

                # Normalize
                passage_embeddings = F.normalize(
                    passage_embeddings,
                    p=2,
                    dim=1
                )

                all_passage_embeddings.append(
                    passage_embeddings.cpu()
                )

    all_passage_embeddings = torch.cat(all_passage_embeddings, dim=0)

    return all_passage_embeddings.numpy().astype(np.float32)

In [21]:
def generate_query_embeddings(model, dataloader, device):

    model.eval()

    all_query_embeddings = []

    with torch.no_grad():
        with torch.amp.autocast("cuda"):

            for batch in tqdm(dataloader, desc="Generating Query Embeddings"):

                query_input_ids = batch["query_input_ids"].to(device)
                query_attention_mask = batch["query_attention_mask"].to(device)

                query_embeddings = model(
                    input_ids=query_input_ids,
                    attention_mask=query_attention_mask
                )

                # Normalize
                query_embeddings = F.normalize(
                    query_embeddings,
                    p=2,
                    dim=1
                )

                all_query_embeddings.append(
                    query_embeddings.cpu()
                )

    all_query_embeddings = torch.cat(all_query_embeddings, dim=0)

    return all_query_embeddings.numpy().astype(np.float32)

In [22]:
def build_faiss_index(passage_embeddings):

    dimension = passage_embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(passage_embeddings)

    print(f"Total Passage Embeddings : {index.ntotal}")

    return index

In [23]:
def retrieve_top_k(index, query_embeddings, top_k=10):

    scores, indices = index.search(
        query_embeddings,
        top_k
    )

    return scores, indices

In [24]:
def recall_at_k(indices, k=10):

    correct = 0
    total_queries = len(indices)

    for query_idx, retrieved_indices in enumerate(indices):

        if query_idx in retrieved_indices[:k]:
            correct += 1

    recall = correct / total_queries

    return recall

In [25]:
import numpy as np

def mean_reciprocal_rank(indices):

    reciprocal_ranks = []

    for query_idx, retrieved_indices in enumerate(indices):

        rr = 0

        for rank, passage_idx in enumerate(retrieved_indices):

            if passage_idx == query_idx:
                rr = 1 / (rank + 1)
                break

        reciprocal_ranks.append(rr)

    return np.mean(reciprocal_ranks)

In [26]:
import math

def ndcg_at_k(indices, k=10):

    ndcg_scores = []

    for query_idx, retrieved_indices in enumerate(indices):

        dcg = 0

        for rank, passage_idx in enumerate(retrieved_indices[:k]):

            if passage_idx == query_idx:

                dcg = 1 / math.log2(rank + 2)
                break

        idcg = 1

        ndcg_scores.append(dcg / idcg)

    return np.mean(ndcg_scores)

In [27]:
def evaluate_retrieval(query_embeddings, passage_embeddings, top_k=10):

    index = build_faiss_index(passage_embeddings)

    scores, indices = retrieve_top_k(
        index,
        query_embeddings,
        top_k
    )

    recall = recall_at_k(indices, top_k)

    mrr = mean_reciprocal_rank(indices)

    ndcg = ndcg_at_k(indices, top_k)

    return recall, mrr, ndcg

In [28]:
query_embeddings = generate_query_embeddings(
    bert_model,
    val_dataLoader,
    device
)

passage_embeddings = generate_passage_embeddings(
    bert_model,
    val_dataLoader,
    device
)

Generating Passage Embeddings: 100%|██████████| 304/304 [00:19<00:00, 15.32it/s]


In [29]:
# recall, mrr, ndcg = evaluate_retrieval(
#     query_embeddings,
#     passage_embeddings,
#     top_k=10
# )

# print(f"Recall@10 : {recall:.4f}")
# print(f"MRR        : {mrr:.4f}")
# print(f"nDCG@10    : {ndcg:.4f}")

In [30]:
def evaluate_model(model,dataloader,device,top_k=10):

    #----------------Embeddings----------------#
    query_embeddings = generate_query_embeddings(model,dataloader,device)

    passage_embeddings = generate_passage_embeddings(model,dataloader,device)

    #----------------FAISS----------------#
    index = build_faiss_index(passage_embeddings)

    scores, indices = retrieve_top_k(index,query_embeddings,top_k)

    #----------------Metrics----------------#
    recall = recall_at_k(indices,top_k)

    mrr = mean_reciprocal_rank(indices)

    ndcg = ndcg_at_k(indices,top_k)

    return {
        "recall": recall,
        "mrr": mrr,
        "ndcg": ndcg,
        "scores": scores,
        "indices": indices,
        "query_embeddings": query_embeddings,
        "passage_embeddings": passage_embeddings
    }

In [31]:
def inspect_retrieval(metrics,queries,passages,query_id,inspect=False):
    print("=" * 100)

    print("Query:\n")
    print(queries[query_id])

    print("\n" + "=" * 100)

    print("Ground Truth Passage:\n")
    print(passages[query_id])

    print("\n" + "=" * 100)

    print("Top Retrieved Passages:\n")

    retrieved_indices = metrics["indices"][query_id]
    scores = metrics["scores"][query_id]

    for rank, (idx, score) in enumerate(zip(retrieved_indices, scores), start=1):

        print(f"\nRank {rank}")
        print(f"Similarity : {score:.4f}")

        if idx == query_id:
            print("✅ Correct")
        else:
            print("❌ Incorrect")

        print(passages[idx])

        print("-" * 80)

In [33]:
best_recall = -1.0
for epoch in range(epochs):
  #-------------Traning loop---------!
  bert_model.train()
  train_loss=0
  progress_bar_train = tqdm(train_dataLoader, desc=f'Train-Epoch {epoch+1}')

  for batch in progress_bar_train:
    query_input_ids, query_attention_mask = batch["query_input_ids"].to(device), batch["query_attention_mask"].to(device)
    passage_input_ids, passage_attention_mask = batch["passage_input_ids"].to(device), batch["passage_attention_mask"].to(device)

    optimizer.zero_grad()
    with torch.amp.autocast('cuda'):
      query_output = bert_model(input_ids=query_input_ids,attention_mask=query_attention_mask)
      passage_output = bert_model(input_ids=passage_input_ids,attention_mask=passage_attention_mask)

      #----Normalize----!
      query_embeddings = F.normalize(query_output,p=2,dim=1)
      passage_embeddings = F.normalize(passage_output,p=2,dim=1)

      #------Cosine Similarity-----!
      similarity = (query_embeddings @ passage_embeddings.T) / temperature

      labels = torch.arange(similarity.size(0),device=device)
      loss = loss_fn(similarity,labels)

    scalar.scale(loss).backward()
    scalar.step(optimizer)
    scalar.update()

    train_loss += loss.item()
    progress_bar_train.set_postfix(loss=f"{loss.item():.6f}")

  train_loss = train_loss / len(train_dataLoader)


  #-------------Validation loop---------!
  bert_model.eval()
  val_loss=0
  progress_bar_val = tqdm(val_dataLoader, desc=f'Validation-Epoch {epoch+1}')

  with torch.no_grad():
    with torch.amp.autocast("cuda"):
      for batch in progress_bar_val:
        query_input_ids, query_attention_mask = batch["query_input_ids"].to(device), batch["query_attention_mask"].to(device)
        passage_input_ids, passage_attention_mask = batch["passage_input_ids"].to(device), batch["passage_attention_mask"].to(device)

        query_output = bert_model(input_ids=query_input_ids,attention_mask=query_attention_mask)
        passage_output = bert_model(input_ids=passage_input_ids,attention_mask=passage_attention_mask)

        #----Normalize----!
        query_embeddings = F.normalize(query_output,p=2,dim=1)
        passage_embeddings = F.normalize(passage_output,p=2,dim=1)

        #------Cosine Similarity-----!
        similarity = (query_embeddings @ passage_embeddings.T) / temperature

        labels = torch.arange(similarity.size(0),device=device)
        loss = loss_fn(similarity,labels)
        val_loss += loss.item()
        progress_bar_val.set_postfix(loss=f"{loss.item():.6f}")

  val_loss = val_loss / len(val_dataLoader)

  print(f"Epoch {epoch+1}")
  print(f"Train Loss : {train_loss:.4f}")
  print(f"Validation Loss : {val_loss:.4f}")
  print("="*50)

  #----------------Retrieval Evaluation----------------#

  metrics = evaluate_model(bert_model,val_dataLoader,device,top_k=10)

  #--- Set true below to print the retrieved passage ---!
  # inspect_retrieval(metrics,val_processed_queries,val_processed_positive_passage,25)

  print(f"\nEpoch {epoch+1}")
  print(f"Train Loss : {train_loss:.4f}")
  print(f"Validation Loss : {val_loss:.4f}")

  print(f"Recall@10 : {metrics['recall']:.4f}")
  print(f"MRR        : {metrics['mrr']:.4f}")
  print(f"nDCG@10    : {metrics['ndcg']:.4f}")

  print("="*60)

  #----------------Save Best Model----------------#

  if metrics["recall"] > best_recall:

      best_recall = metrics["recall"]

      torch.save(
          {
              "epoch": epoch + 1,
              "model_state_dict": bert_model.state_dict(),
              "optimizer_state_dict": optimizer.state_dict(),
              "recall": metrics["recall"],
              "mrr": metrics["mrr"],
              "ndcg": metrics["ndcg"]
          },
          "best_model.pth"
        )

      print("✅ Best model saved!")

Validation-Epoch 1: 100%|██████████| 304/304 [00:24<00:00, 12.19it/s, loss=0.000023]


Epoch 1
Train Loss : 0.0201
Validation Loss : 0.0246


Generating Passage Embeddings: 100%|██████████| 304/304 [00:21<00:00, 14.27it/s]


Total Passage Embeddings : 9706

Epoch 1
Train Loss : 0.0201
Validation Loss : 0.0246
Recall@10 : 0.9667
MRR        : 0.8456
nDCG@10    : 0.8756
✅ Best model saved!


Validation-Epoch 2: 100%|██████████| 304/304 [00:24<00:00, 12.21it/s, loss=0.000040]


...

Total Passage Embeddings : 9706

Epoch 9
Train Loss : 0.0058
Validation Loss : 0.0245
Recall@10 : 0.9662
MRR        : 0.8524
nDCG@10    : 0.8805


Validation-Epoch 10: 100%|██████████| 304/304 [00:24<00:00, 12.31it/s, loss=0.000009]


Epoch 10
Train Loss : 0.0053
Validation Loss : 0.0285


Generating Passage Embeddings: 100%|██████████| 304/304 [00:21<00:00, 14.21it/s]


Total Passage Embeddings : 9706

Epoch 10
Train Loss : 0.0053
Validation Loss : 0.0285
Recall@10 : 0.9656
MRR        : 0.8468
nDCG@10    : 0.8762


# **Evaluation on Test Data**

In [35]:
checkpoint = torch.load("best_model.pth",map_location=device,weights_only=False)

bert_model.load_state_dict(checkpoint["model_state_dict"])
bert_model.eval()

metrics = evaluate_model(bert_model,val_dataLoader,device,top_k=10)

print(f"Recall@10 : {metrics['recall']:.4f}")
print(f"MRR        : {metrics['mrr']:.4f}")
print(f"nDCG@10    : {metrics['ndcg']:.4f}")

Generating Passage Embeddings: 100%|██████████| 304/304 [00:20<00:00, 15.15it/s]


Total Passage Embeddings : 9706
Recall@10 : 0.9693
MRR        : 0.8521
nDCG@10    : 0.8810


In [38]:
# train_eval_dataLoader = DataLoader(
#     train_dataset,
#     batch_size=32,
#     shuffle=False,
#     collate_fn=retrieval_collate_fn
# )

In [ ]:
# metrics = evaluate_model(
#     bert_model,
#     train_eval_dataLoader,
#     device,
#     top_k=10
# )
# print(f"Recall@10 : {metrics['recall']:.4f}")
# print(f"MRR        : {metrics['mrr']:.4f}")
# print(f"nDCG@10    : {metrics['ndcg']:.4f}")